In [3]:
from matplotlib import pyplot as plt
import numpy as np
import collections
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm.notebook import tqdm

In [9]:
# device = 'cpu'
device = 'cuda'

In [13]:
torch.set_default_dtype(torch.float32)
batch_size, seqlen, i_dim, hidden = 2048,128,38,32

print(torch.cuda.get_device_name())
print('torch:', torch.__version__)
net = nn.Sequential(nn.Linear(i_dim, hidden), nn.Tanh(), nn.Linear(hidden, 1)).to(device)
print('params count:', sum([p.numel() for p in net.parameters()]))

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
X_b = torch.randn((batch_size,seqlen,i_dim), device=device)
y_true = torch.randn((batch_size,seqlen,1), device=device)
# loss_fn = F.mse_loss
loss_fn = F.cross_entropy

for epoch in range(4):
    t_forward=0
    t_backward=0
    
    for _ in range(100):
        optimizer.zero_grad(set_to_none=True)
        
        t0=time.time()
        pred = net(X_b)
        loss = loss_fn(pred, y_true)
        torch.cuda.synchronize()
        
        t1=time.time()
        loss.backward()
        
        torch.cuda.synchronize()
        t2=time.time()
        
        t_forward+=(t1-t0)
        t_backward+=(t2-t1)
        
    print(f"{epoch}: forward: {t_forward:.2f}s, backward: {t_backward:.2f}s")

NVIDIA GeForce RTX 4080 SUPER
torch: 2.9.1+cu128
params count: 1281
0: forward: 0.19s, backward: 0.18s
1: forward: 0.10s, backward: 0.17s
2: forward: 0.10s, backward: 0.17s
3: forward: 0.10s, backward: 0.17s


In [20]:
class TestModel(nn.Module):
    def __init__(self, inp_dims_count, hiddens_count):
        super().__init__()
        self.filters = nn.Linear(inp_dims_count, hiddens_count)
        self.decoder = nn.Linear(hiddens_count, inp_dims_count)

        bound = 1 / np.sqrt(inp_dims_count)
        self.filters.weight.data.uniform_(-bound, bound)
        bound = 1 / np.sqrt(hiddens_count)
        self.decoder.weight.data.uniform_(-bound, bound)

    def forward(self, inp):
        out = F.sigmoid(self.filters(inp))
        out = self.decoder(out)
        return out

In [26]:
torch.set_default_dtype(torch.float32)

batch_size, i_dim, hidden = 2048, 144, 200

net = TestModel(144, 200).to(device=device)
print('params count:', sum([p.numel() for p in net.parameters()]))

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
X_b = torch.randn((batch_size, i_dim), device=device)
y_true = torch.randn((batch_size, i_dim), device=device)
loss_fn = F.mse_loss

for epoch in range(4):
    t_forward=0
    t_backward=0
    
    for _ in range(100):
        optimizer.zero_grad(set_to_none=True)
        
        t0=time.time()
        pred = net(X_b)
        loss = loss_fn(pred, y_true)
        torch.cuda.synchronize()
        
        t1=time.time()
        loss.backward()
        
        torch.cuda.synchronize()
        t2=time.time()
        
        t_forward+=(t1-t0)
        t_backward+=(t2-t1)
        
    print(f"{epoch}: forward: {t_forward:.2f}s, backward: {t_backward:.2f}s")

params count: 57944
0: forward: 0.08s, backward: 0.14s
1: forward: 0.08s, backward: 0.14s
2: forward: 0.08s, backward: 0.14s
3: forward: 0.08s, backward: 0.14s
